To create history and edtect for anamoly we need multiple days data, lets do that in same medallian structure 

In [0]:
%skip
spark.sql("""
ALTER TABLE workspace.bronze.events_raw
ADD COLUMNS (source_file STRING)
""")

In [0]:
import requests

files = [
    "20260911.export.CSV.zip",
    "20260910.export.CSV.zip",
    "20260909.export.CSV.zip"
]

base_url = "https://data.gdeltproject.org/events/"

for filename in files:
    url = base_url + filename
    local_path = f"/tmp/{filename}"

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(local_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {filename}")

In [0]:
import shutil

for filename in files:
    local_path = f"/tmp/{filename}"
    destination = f"/Volumes/workspace/bronze/gdelt_raw/{filename}"

    shutil.copy(local_path, destination)

    print(f"Stored: {destination}")

In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/bronze/gdelt_raw/")
)

In [0]:
import zipfile
import os
import shutil
import pandas as pd
from pyspark.sql import functions as F

gdelt_columns = [
    "GlobalEventID",
    "SQLDATE",
    "MonthYear",
    "Year",
    "FractionDate",
    "Actor1Code",
    "Actor1Name",
    "Actor1CountryCode",
    "Actor1KnownGroupCode",
    "Actor1EthnicCode",
    "Actor1Religion1Code",
    "Actor1Religion2Code",
    "Actor1Type1Code",
    "Actor1Type2Code",
    "Actor1Type3Code",
    "Actor2Code",
    "Actor2Name",
    "Actor2CountryCode",
    "Actor2KnownGroupCode",
    "Actor2EthnicCode",
    "Actor2Religion1Code",
    "Actor2Religion2Code",
    "Actor2Type1Code",
    "Actor2Type2Code",
    "Actor2Type3Code",
    "IsRootEvent",
    "EventCode",
    "EventBaseCode",
    "EventRootCode",
    "QuadClass",
    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",
    "Actor1Geo_Type",
    "Actor1Geo_FullName",
    "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code",
    "Actor1Geo_Lat",
    "Actor1Geo_Long",
    "Actor1Geo_FeatureID",
    "Actor2Geo_Type",
    "Actor2Geo_FullName",
    "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code",
    "Actor2Geo_Lat",
    "Actor2Geo_Long",
    "Actor2Geo_FeatureID",
    "ActionGeo_Type",
    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code",
    "ActionGeo_Lat",
    "ActionGeo_Long",
    "ActionGeo_FeatureID",
    "DATEADDED",
    "SOURCEURL"
]

def read_gdelt_zip(filename):
    volume_path = f"/Volumes/workspace/bronze/gdelt_raw/{filename}"
    local_zip = f"/tmp/{filename}"

    shutil.copy(volume_path, local_zip)

    extract_dir = f"/tmp/{filename.replace('.zip', '')}"
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(local_zip, "r") as z:
        z.extractall(extract_dir)

    csv_files = [
        os.path.join(extract_dir, f)
        for f in os.listdir(extract_dir)
        if f.endswith(".CSV")
    ]

    if not csv_files:
        raise ValueError(f"No CSV found inside {filename}")

    csv_path = csv_files[0]

    pdf = pd.read_csv(csv_path, sep="\t", header=None, names=gdelt_columns, dtype=str)
    df = spark.createDataFrame(pdf).withColumn("source_file", F.lit(filename))

    return df

In [0]:
df_test = read_gdelt_zip("20260911.export.CSV.zip")

print("Rows:", df_test.count())
display(df_test.select(
    "GlobalEventID",
    "SQLDATE",
    "ActionGeo_CountryCode",
    "source_file"
).limit(5))

In [0]:
(
    df_test.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.bronze.events_raw")
)

In [0]:
display(
    spark.sql("""
        SELECT
            source_file,
            COUNT(*) AS event_count
        FROM workspace.bronze.events_raw
        GROUP BY source_file
        ORDER BY source_file DESC
    """)
)

In [0]:
for filename in [
    "20260910.export.CSV.zip",
    "20260909.export.CSV.zip"
]:
    df_new = read_gdelt_zip(filename)

    row_count = df_new.count()

    (
        df_new.write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.bronze.events_raw")
    )

    print(f"Loaded {filename}: {row_count:,} rows")

In [0]:
display(
    spark.sql("""
        SELECT
            source_file,
            COUNT(*) AS event_count
        FROM workspace.bronze.events_raw
        GROUP BY source_file
        ORDER BY source_file DESC
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            source_file,
            COUNT(*) AS rows
        FROM workspace.bronze.events_raw
        GROUP BY source_file
        ORDER BY source_file DESC
    """)
)

In [0]:
silver_candidate = spark.sql("""
SELECT
    try_cast(GlobalEventID AS BIGINT) AS GlobalEventID,

    try_to_date(SQLDATE, 'yyyyMMdd') AS EventDate,
    try_cast(MonthYear AS INT) AS MonthYear,
    try_cast(Year AS INT) AS Year,
    try_cast(FractionDate AS DOUBLE) AS FractionDate,

    Actor1Code,
    Actor1Name,
    Actor1CountryCode,
    Actor1KnownGroupCode,
    Actor1EthnicCode,
    Actor1Religion1Code,
    Actor1Religion2Code,
    Actor1Type1Code,
    Actor1Type2Code,
    Actor1Type3Code,

    Actor2Code,
    Actor2Name,
    Actor2CountryCode,
    Actor2KnownGroupCode,
    Actor2EthnicCode,
    Actor2Religion1Code,
    Actor2Religion2Code,
    Actor2Type1Code,
    Actor2Type2Code,
    Actor2Type3Code,

    try_cast(IsRootEvent AS INT) AS IsRootEvent,

    EventCode,
    EventBaseCode,
    EventRootCode,

    try_cast(QuadClass AS INT) AS QuadClass,
    try_cast(GoldsteinScale AS DOUBLE) AS GoldsteinScale,
    try_cast(NumMentions AS INT) AS NumMentions,
    try_cast(NumSources AS INT) AS NumSources,
    try_cast(NumArticles AS INT) AS NumArticles,
    try_cast(AvgTone AS DOUBLE) AS AvgTone,

    try_cast(Actor1Geo_Type AS INT) AS Actor1Geo_Type,
    Actor1Geo_FullName,
    Actor1Geo_CountryCode,
    Actor1Geo_ADM1Code,
    try_cast(Actor1Geo_Lat AS DOUBLE) AS Actor1Geo_Lat,
    try_cast(Actor1Geo_Long AS DOUBLE) AS Actor1Geo_Long,
    Actor1Geo_FeatureID,

    try_cast(Actor2Geo_Type AS INT) AS Actor2Geo_Type,
    Actor2Geo_FullName,
    Actor2Geo_CountryCode,
    Actor2Geo_ADM1Code,
    try_cast(Actor2Geo_Lat AS DOUBLE) AS Actor2Geo_Lat,
    try_cast(Actor2Geo_Long AS DOUBLE) AS Actor2Geo_Long,
    Actor2Geo_FeatureID,

    try_cast(ActionGeo_Type AS INT) AS ActionGeo_Type,
    ActionGeo_FullName,
    ActionGeo_CountryCode,
    ActionGeo_ADM1Code,
    try_cast(ActionGeo_Lat AS DOUBLE) AS ActionGeo_Lat,
    try_cast(ActionGeo_Long AS DOUBLE) AS ActionGeo_Long,
    ActionGeo_FeatureID,

    DATEADDED,
    SOURCEURL,
    source_file

FROM workspace.bronze.events_raw
""")

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

dedup_window = Window.partitionBy("GlobalEventID").orderBy(
    F.col("DATEADDED").desc(),
    F.col("source_file").desc()
)

silver_dedup = (
    silver_candidate
    .withColumn("_row_number", F.row_number().over(dedup_window))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
bronze_count = spark.table("workspace.bronze.events_raw").count()
silver_dedup_count = silver_dedup.count()

print(f"Bronze rows:       {bronze_count:,}")
print(f"After dedup:       {silver_dedup_count:,}")
print(f"Duplicates removed: {bronze_count - silver_dedup_count:,}")

In [0]:
numeric_checks = {
    "FractionDate": "DOUBLE",
    "IsRootEvent": "INT",
    "QuadClass": "INT",
    "GoldsteinScale": "DOUBLE",
    "NumMentions": "INT",
    "NumSources": "INT",
    "NumArticles": "INT",
    "AvgTone": "DOUBLE",
    "Actor1Geo_Type": "INT",
    "Actor1Geo_Lat": "DOUBLE",
    "Actor1Geo_Long": "DOUBLE",
    "Actor2Geo_Type": "INT",
    "Actor2Geo_Lat": "DOUBLE",
    "Actor2Geo_Long": "DOUBLE",
    "ActionGeo_Type": "INT",
    "ActionGeo_Lat": "DOUBLE",
    "ActionGeo_Long": "DOUBLE"
}

for column, dtype in numeric_checks.items():
    bad = spark.sql(f"""
        SELECT COUNT(*) AS bad_count
        FROM workspace.bronze.events_raw
        WHERE {column} IS NOT NULL
          AND try_cast({column} AS {dtype}) IS NULL
    """).collect()[0]["bad_count"]

    if bad > 0:
        print(f"{column}: {bad:,} malformed values")

In [0]:
silver_checked = (
    silver_dedup

    .withColumn(
        "is_valid_event_id",
        F.col("GlobalEventID").isNotNull()
    )

    .withColumn(
        "is_valid_date",
        F.col("EventDate").isNotNull()
    )

    .withColumn(
        "is_valid_quad_class",
        F.col("QuadClass").isNull() |
        F.col("QuadClass").between(1, 4)
    )

    .withColumn(
        "is_valid_action_lat",
        F.col("ActionGeo_Lat").isNull() |
        F.col("ActionGeo_Lat").between(-90, 90)
    )

    .withColumn(
        "is_valid_action_long",
        F.col("ActionGeo_Long").isNull() |
        F.col("ActionGeo_Long").between(-180, 180)
    )
)

In [0]:
silver_checked = silver_checked.withColumn(
    "is_valid_record",
    F.col("is_valid_event_id")
    & F.col("is_valid_date")
    & F.col("is_valid_quad_class")
    & F.col("is_valid_action_lat")
    & F.col("is_valid_action_long")
)

In [0]:
total = silver_checked.count()

invalid = (
    silver_checked
    .filter(~F.col("is_valid_record"))
    .count()
)

print(f"Total records:   {total:,}")
print(f"Invalid records: {invalid:,}")
print(f"Invalid %:       {(invalid / total) * 100:.4f}%")

In [0]:
numeric_checks = {
    "FractionDate": "DOUBLE",
    "IsRootEvent": "INT",
    "QuadClass": "INT",
    "GoldsteinScale": "DOUBLE",
    "NumMentions": "INT",
    "NumSources": "INT",
    "NumArticles": "INT",
    "AvgTone": "DOUBLE",
    "Actor1Geo_Type": "INT",
    "Actor1Geo_Lat": "DOUBLE",
    "Actor1Geo_Long": "DOUBLE",
    "Actor2Geo_Type": "INT",
    "Actor2Geo_Lat": "DOUBLE",
    "Actor2Geo_Long": "DOUBLE",
    "ActionGeo_Type": "INT",
    "ActionGeo_Lat": "DOUBLE",
    "ActionGeo_Long": "DOUBLE"
}

for column, dtype in numeric_checks.items():
    bad = spark.sql(f"""
        SELECT COUNT(*) AS bad_count
        FROM workspace.bronze.events_raw
        WHERE {column} IS NOT NULL
          AND try_cast({column} AS {dtype}) IS NULL
    """).collect()[0]["bad_count"]

    if bad > 0:
        print(f"{column}: {bad:,} malformed values")

In [0]:
semantic_quality = silver_dedup.selectExpr(
    "COUNT(*) AS total_records",

    """
    SUM(
        CASE WHEN GlobalEventID IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_event_id
    """,

    """
    SUM(
        CASE
            WHEN EventDate IS NULL THEN 1
            ELSE 0
        END
    ) AS invalid_dates
    """,

    """
    SUM(
        CASE
            WHEN QuadClass IS NOT NULL
             AND QuadClass NOT BETWEEN 1 AND 4
            THEN 1 ELSE 0
        END
    ) AS invalid_quad_class
    """,

    """
    SUM(
        CASE
            WHEN ActionGeo_Lat IS NOT NULL
             AND (ActionGeo_Lat < -90 OR ActionGeo_Lat > 90)
            THEN 1 ELSE 0
        END
    ) AS invalid_action_lat
    """,

    """
    SUM(
        CASE
            WHEN ActionGeo_Long IS NOT NULL
             AND (ActionGeo_Long < -180 OR ActionGeo_Long > 180)
            THEN 1 ELSE 0
        END
    ) AS invalid_action_long
    """
)

display(semantic_quality)

In [0]:
silver_events = (
    silver_dedup
    .filter(F.col("GlobalEventID").isNotNull())
    .filter(F.col("EventDate").isNotNull())
    .filter(
        F.col("QuadClass").isNull() |
        F.col("QuadClass").between(1, 4)
    )
    .filter(
        F.col("ActionGeo_Lat").isNull() |
        F.col("ActionGeo_Lat").between(-90, 90)
    )
    .filter(
        F.col("ActionGeo_Long").isNull() |
        F.col("ActionGeo_Long").between(-180, 180)
    )
)

In [0]:
(
    silver_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.events")
)

In [0]:
silver_count = spark.table(
    "workspace.silver.events"
).count()

print(f"Silver records: {silver_count:,}")

In [0]:
bronze_count = spark.table(
    "workspace.bronze.events_raw"
).count()

print(f"Bronze records: {bronze_count:,}")
print(f"Removed during processing: {bronze_count - silver_count:,}")

In [0]:
display(
    spark.sql("""
        SELECT
            EventDate,
            COUNT(*) AS EventCount
        FROM workspace.silver.events
        GROUP BY EventDate
        ORDER BY EventDate
    """)
)

In [0]:
gold_country_daily = spark.sql("""
SELECT
    EventDate,
    ActionGeo_CountryCode AS CountryCode,
    COUNT(*) AS EventCount,
    SUM(NumMentions) AS TotalMentions,
    SUM(NumSources) AS TotalSources,
    SUM(NumArticles) AS TotalArticles,
    AVG(GoldsteinScale) AS AvgGoldsteinScale,
    AVG(AvgTone) AS AvgTone
FROM workspace.silver.events
WHERE ActionGeo_CountryCode IS NOT NULL
GROUP BY
    EventDate,
    ActionGeo_CountryCode
""")

In [0]:
(
    gold_country_daily.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.country_daily")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.gold.country_daily
        ORDER BY EventDate, EventCount DESC
    """).limit(30)
)

In [0]:
display(
    spark.sql("""
        SELECT
            COALESCE(source_file, 'NULL_SOURCE_FILE') AS source_file,
            COUNT(*) AS row_count,
            COUNT(DISTINCT GlobalEventID) AS unique_event_ids
        FROM workspace.bronze.events_raw
        GROUP BY COALESCE(source_file, 'NULL_SOURCE_FILE')
        ORDER BY source_file
    """)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.table("workspace.bronze.events_raw")

dedup_window = Window.partitionBy("GlobalEventID").orderBy(
    F.when(F.col("source_file").isNotNull(), 0).otherwise(1),
    F.col("source_file")
)

bronze_clean = (
    bronze
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print("Original Bronze rows:", bronze.count())
print("Clean Bronze rows:", bronze_clean.count())
print(
    "Duplicates removed:",
    bronze.count() - bronze_clean.count()
)

In [0]:
duplicate_check = (
    bronze_clean
    .groupBy("GlobalEventID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate GlobalEventIDs after cleanup:", duplicate_check.count())
display(duplicate_check)

In [0]:
display(
    bronze_clean
    .groupBy("source_file")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("GlobalEventID").alias("unique_event_ids")
    )
    .orderBy("source_file")
)

In [0]:
bronze_clean = bronze_clean.withColumn(
    "source_file",
    F.when(
        F.col("source_file").isNull(),
        F.lit("20260912.export.CSV.zip")
    ).otherwise(F.col("source_file"))
)

In [0]:
display(
    bronze_clean
    .groupBy("source_file")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("GlobalEventID").alias("unique_event_ids")
    )
    .orderBy("source_file")
)

In [0]:
(
    bronze_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.events_raw_clean")
)

In [0]:
print(
    spark.table("workspace.bronze.events_raw_clean").count()
)

In [0]:
display(
    spark.sql("""
        SELECT
            source_file,
            COUNT(*) AS rows,
            COUNT(DISTINCT GlobalEventID) AS unique_event_ids
        FROM workspace.bronze.events_raw_clean
        GROUP BY source_file
        ORDER BY source_file
    """)
)

In [0]:
silver_candidate = spark.sql("""
SELECT
    try_cast(GlobalEventID AS BIGINT) AS GlobalEventID,

    try_to_date(SQLDATE, 'yyyyMMdd') AS EventDate,
    try_cast(MonthYear AS INT) AS MonthYear,
    try_cast(Year AS INT) AS Year,
    try_cast(FractionDate AS DOUBLE) AS FractionDate,

    Actor1Code,
    Actor1Name,
    Actor1CountryCode,
    Actor1KnownGroupCode,
    Actor1EthnicCode,
    Actor1Religion1Code,
    Actor1Religion2Code,
    Actor1Type1Code,
    Actor1Type2Code,
    Actor1Type3Code,

    Actor2Code,
    Actor2Name,
    Actor2CountryCode,
    Actor2KnownGroupCode,
    Actor2EthnicCode,
    Actor2Religion1Code,
    Actor2Religion2Code,
    Actor2Type1Code,
    Actor2Type2Code,
    Actor2Type3Code,

    try_cast(IsRootEvent AS INT) AS IsRootEvent,

    EventCode,
    EventBaseCode,
    EventRootCode,

    try_cast(QuadClass AS INT) AS QuadClass,
    try_cast(GoldsteinScale AS DOUBLE) AS GoldsteinScale,
    try_cast(NumMentions AS INT) AS NumMentions,
    try_cast(NumSources AS INT) AS NumSources,
    try_cast(NumArticles AS INT) AS NumArticles,
    try_cast(AvgTone AS DOUBLE) AS AvgTone,

    try_cast(Actor1Geo_Type AS INT) AS Actor1Geo_Type,
    Actor1Geo_FullName,
    Actor1Geo_CountryCode,
    Actor1Geo_ADM1Code,
    try_cast(Actor1Geo_Lat AS DOUBLE) AS Actor1Geo_Lat,
    try_cast(Actor1Geo_Long AS DOUBLE) AS Actor1Geo_Long,
    Actor1Geo_FeatureID,

    try_cast(Actor2Geo_Type AS INT) AS Actor2Geo_Type,
    Actor2Geo_FullName,
    Actor2Geo_CountryCode,
    Actor2Geo_ADM1Code,
    try_cast(Actor2Geo_Lat AS DOUBLE) AS Actor2Geo_Lat,
    try_cast(Actor2Geo_Long AS DOUBLE) AS Actor2Geo_Long,
    Actor2Geo_FeatureID,

    try_cast(ActionGeo_Type AS INT) AS ActionGeo_Type,
    ActionGeo_FullName,
    ActionGeo_CountryCode,
    ActionGeo_ADM1Code,
    try_cast(ActionGeo_Lat AS DOUBLE) AS ActionGeo_Lat,
    try_cast(ActionGeo_Long AS DOUBLE) AS ActionGeo_Long,
    ActionGeo_FeatureID,

    DATEADDED,
    SOURCEURL,
    source_file

FROM workspace.bronze.events_raw_clean
""")

In [0]:
silver_events = (
    silver_candidate
    .filter(F.col("GlobalEventID").isNotNull())
    .filter(F.col("EventDate").isNotNull())
    .filter(
        F.col("QuadClass").isNull() |
        F.col("QuadClass").between(1, 4)
    )
    .filter(
        F.col("ActionGeo_Lat").isNull() |
        F.col("ActionGeo_Lat").between(-90, 90)
    )
    .filter(
        F.col("ActionGeo_Long").isNull() |
        F.col("ActionGeo_Long").between(-180, 180)
    )
)

In [0]:
print("Silver records:", silver_events.count())

In [0]:
(
    silver_events.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.events")
)

In [0]:
display(
    spark.sql("""
        SELECT
            EventDate,
            COUNT(*) AS events
        FROM workspace.silver.events
        GROUP BY EventDate
        ORDER BY EventDate
    """)
)

In [0]:
silver_events = (
    spark.table("workspace.silver.events")
    .withColumn(
        "AddedTimestamp",
        F.try_to_timestamp(
            F.col("DATEADDED"),
            F.lit("yyyyMMddHHmmss")
        )
    )
    .withColumn(
        "AddedDate",
        F.to_date("AddedTimestamp")
    )
)

In [0]:
display(
    silver_events.select(
        "GlobalEventID",
        "EventDate",
        "AddedTimestamp",
        "AddedDate",
        "source_file"
    ).orderBy("AddedTimestamp").limit(20)
)

In [0]:
gold_country_ingestion_daily = (
    silver_events
    .filter(F.col("AddedDate").isNotNull())
    .filter(F.col("ActionGeo_CountryCode").isNotNull())
    .groupBy(
        "AddedDate",
        F.col("ActionGeo_CountryCode").alias("CountryCode")
    )
    .agg(
        F.count("*").alias("EventCount"),
        F.sum("NumMentions").alias("TotalMentions"),
        F.sum("NumSources").alias("TotalSources"),
        F.sum("NumArticles").alias("TotalArticles"),
        F.avg("GoldsteinScale").alias("AvgGoldsteinScale"),
        F.avg("AvgTone").alias("AvgTone")
    )
)

In [0]:
(
    gold_country_ingestion_daily.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.country_ingestion_daily")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.gold.country_ingestion_daily
        ORDER BY AddedDate, EventCount DESC
    """).limit(50)
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.gold.country_ingestion_daily
        ORDER BY AddedDate, EventCount DESC
    """).limit(50)
)

In [0]:
from pyspark.sql import functions as F

silver_events = (
    spark.table("workspace.silver.events")
    .withColumn(
        "AddedDate",
        F.to_date("DATEADDED", "yyyyMMdd")
    )
)

In [0]:
display(
    silver_events.select(
        "GlobalEventID",
        "EventDate",
        "DATEADDED",
        "AddedDate",
        "source_file"
    ).limit(20)
)

In [0]:
silver_events.select(
    F.count("*").alias("total"),
    F.count("AddedDate").alias("parsed_added_dates")
).show()

In [0]:
gold_country_ingestion_daily = (
    silver_events
    .filter(F.col("AddedDate").isNotNull())
    .filter(F.col("ActionGeo_CountryCode").isNotNull())
    .groupBy(
        "AddedDate",
        F.col("ActionGeo_CountryCode").alias("CountryCode")
    )
    .agg(
        F.count("*").alias("EventCount"),
        F.sum("NumMentions").alias("TotalMentions"),
        F.sum("NumSources").alias("TotalSources"),
        F.sum("NumArticles").alias("TotalArticles"),
        F.avg("GoldsteinScale").alias("AvgGoldsteinScale"),
        F.avg("AvgTone").alias("AvgTone")
    )
)

In [0]:
(
    gold_country_ingestion_daily.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.country_ingestion_daily")
)

In [0]:
display(
    gold_country_ingestion_daily
    .orderBy("AddedDate", F.col("EventCount").desc())
    .limit(50)
)